# 관계를 숫자로

> 파이썬 12강 · 통계적 추론

이 노트북은 웹 강의의 **실습 부분만** 옮겨온 것입니다.
자세한 설명과 그림은 원문을 함께 보세요 → [관계를 숫자로](https://mioon1402.github.io/timeseriesdata/python/p12-regression.html)

---

**먼저 아래 준비 셀을 한 번 실행하세요.** 예시 데이터를 내려받습니다.

In [ ]:
# 예시 데이터 내려받기
!wget -q -nc https://raw.githubusercontent.com/mioon1402/timeseriesdata/main/data/cafe_sales.csv

# 표를 글자로 찍을 때 한글 열이 어긋나지 않게 (한글을 두 칸으로 계산)
import pandas as pd
pd.set_option("display.unicode.east_asian_width", True)

# 그래프 한글 깨짐 방지
!pip install -q koreanize-matplotlib
import koreanize_matplotlib  # noqa: F401

print('준비 완료')

## 1. 상관계수

**12-1. 두 종류의 상관계수**

In [ ]:
import pandas as pd
from scipy import stats

df = pd.read_csv("cafe_sales.csv", parse_dates=["date"])
d = df.dropna(subset=["avg_temp", "sales"])

피어슨 = d["avg_temp"].corr(d["sales"])
스피어만 = d["avg_temp"].corr(d["sales"], method="spearman")

print(f"피어슨   r = {피어슨:.4f}   (직선 관계의 강도)")
print(f"스피어만 ρ = {스피어만:.4f}   (순위 관계의 강도)")
print()

r, p = stats.pearsonr(d["avg_temp"], d["sales"])
print(f"scipy: r={r:.4f},  p={p:.3g}")
print(f"결정계수 r² = {r**2:.4f}  →  매출 변동의 {r**2*100:.1f}%를 기온으로 설명")

## 2. 회귀 — 관계를 식으로

**12-2. 가장 간단한 회귀**

In [ ]:
결과 = stats.linregress(d["avg_temp"], d["sales"])

print(f"기울기: {결과.slope:>12,.0f} 원/℃")
print(f"절편:   {결과.intercept:>12,.0f} 원")
print(f"r:      {결과.rvalue:>12.4f}")
print(f"r²:     {결과.rvalue**2:>12.4f}")
print(f"p:      {결과.pvalue:>12.3g}")
print()
print(f"식:  매출 = {결과.intercept:,.0f} + {결과.slope:,.0f} × 기온")

## 3. statsmodels 결과표 읽는 법

**12-3. OLS 회귀**

In [ ]:
import statsmodels.api as sm

X = sm.add_constant(d["avg_temp"])     # 절편을 위한 상수항 추가
모델 = sm.OLS(d["sales"], X).fit()

print(모델.summary().tables[1])
print()
print(f"R² = {모델.rsquared:.4f}   관측수 = {int(모델.nobs)}")

## 4. 잔차 진단 — 모델을 믿기 전에

**12-4. 잔차 그림**

In [ ]:
import matplotlib.pyplot as plt

예측 = 모델.fittedvalues
잔차 = 모델.resid

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.6))

ax1.scatter(예측 / 10000, 잔차 / 10000, s=8, alpha=0.35)
ax1.axhline(0, color="red", linestyle="--", linewidth=1)
ax1.set_xlabel("예측값(만원)"); ax1.set_ylabel("잔차(만원)")
ax1.set_title("잔차 vs 예측값")

ax2.hist(잔차 / 10000, bins=40, edgecolor="white")
ax2.set_xlabel("잔차(만원)"); ax2.set_title("잔차 분포")

plt.tight_layout()
plt.show()

## 5. 다중회귀 — 변수를 여러 개

**12-5. 변수 세 개로**

In [ ]:
d2 = d.copy()
d2["주말"] = (d2["date"].dt.dayofweek >= 5).astype(int)

X2 = sm.add_constant(d2[["avg_temp", "rain_mm", "주말"]])
모델2 = sm.OLS(d2["sales"], X2).fit()

print(모델2.summary().tables[1])
print()
print(f"R²      {모델2.rsquared:.4f}   (기온만: 0.2315)")
print(f"수정 R² {모델2.rsquared_adj:.4f}")

## 6. 계수를 말로 옮기기

**12-6. 예측해보기**

In [ ]:
import pandas as pd

새로운날 = pd.DataFrame({
    "const":    [1, 1, 1],
    "avg_temp": [5, 20, 30],
    "rain_mm":  [0, 0, 20],
    "주말":      [0, 1, 1],
})

예측값 = 모델2.predict(새로운날)

# 한글 라벨을 공백으로 줄맞춤하면 브라우저에서 열이 어긋난다. 표로 만들면 안전하다.
pd.DataFrame({
    "기온(℃)":  새로운날["avg_temp"],
    "비(mm)":   새로운날["rain_mm"],
    "요일":      새로운날["주말"].map({0: "평일", 1: "주말"}),
    "예측매출":   예측값.round(0),
})

**연습 · 직접 써보세요**

In [ ]:
# 문제 1. 방문객(visitors)을 기온·강수량·주말로 회귀해보세요.
#        R²가 매출 모델과 어떻게 다른가요?


# 문제 2. 이상치(방문객 300명 이상인 날)를 빼고 12-5 모델을 다시 돌려보세요.
#        계수가 얼마나 바뀌나요?


# 문제 3. 강수량 대신 '비가 왔는가(0/1)'를 넣으면 계수가 어떻게 해석되나요?

**모범 답안**

In [ ]:
# 문제 1
m_v = sm.OLS(d2["visitors"], X2).fit()
print("방문객 모델 R²:", round(m_v.rsquared, 4))
print(m_v.summary().tables[1])

# 문제 2
평상시 = d2[d2["visitors"] < 300]
X3 = sm.add_constant(평상시[["avg_temp", "rain_mm", "주말"]])
m3 = sm.OLS(평상시["sales"], X3).fit()
print("\n[이상치 제외]")
print(m3.summary().tables[1])
print("R²:", round(m3.rsquared, 4), " (제외 전 0.3852)")

# 문제 3
d2["비옴"] = (d2["rain_mm"] > 0).astype(int)
X4 = sm.add_constant(d2[["avg_temp", "비옴", "주말"]])
m4 = sm.OLS(d2["sales"], X4).fit()
print("\n[비를 0/1 로]")
print(m4.summary().tables[1])

---

전체 강의 목록 → [눈으로 보는 통계](https://mioon1402.github.io/timeseriesdata/)